# akhet: read hieroglyphs from an image

Page photo in, Gardiner codes, Manuel de Codage, transliteration and translation out, with a human correction possible between every two steps.
In Colab choose a GPU runtime (Runtime, Change runtime type). It also runs on CPU, only slower.

In [ ]:
# install (in Colab). Locally: pip install -e ".[gpu,hub,translate]" from the pipeline folder
!git clone -q https://github.com/AIRI-Institute/akhet
!pip install -q "./akhet/pipeline[gpu,hub,translate]"

In [ ]:
from akhet import Akhet, Page, text_blocks, save_blocks, load_blocks, Translator

MODELS = None                      # None = download from the Hugging Face Hub, or a local folder with the models
IMAGE = "akhet/pipeline/examples/sample_page.png"

ocr = Akhet.from_pretrained(MODELS)
page = ocr(IMAGE)
print(len(page.lines), "lines,", sum(len(l.signs) for l in page.lines), "signs")
page.draw(IMAGE)

## 1. Read the OCR result

Blue: lines (`#` reading order, `b` text block, direction). Green: signs, numbered in reading order with their Gardiner code. Red: modern text.

In [ ]:
for line in page.lines:
    print(f"#{line.page_index} block {line.block} {'row' if line.is_row else 'column'} {line.direction:7s} {line.mdc}")
s = page.lines[0].signs[0]
print("\nfirst sign:", s.code, round(s.similarity, 3), "alternatives:", [(c, round(v, 3)) for c, v in s.candidates])

## 2. Correct the OCR by hand

Save the page as JSON and edit it anywhere, or edit it here. `relayout()` recomputes the reading order, the Manuel de Codage and the text blocks from your corrections.

In [ ]:
page.to_json("page.json")
page = Page.from_json("page.json")

line = page.lines[0]
print("before:", line.mdc)
line.signs[0].code = line.signs[0].candidates[1][0]     # e.g. take the second suggestion for the first sign
page.relayout()
print("after: ", page.lines[0].mdc)

## 3. Transliterate, correct, translate

The text stage works on text blocks. Each block is plain editable text. A stage only fills empty fields, so your corrections survive a re-run.

In [ ]:
blocks = text_blocks(page)
for b in blocks:
    print(f"block {b.block}: {b.hieroglyphs}")

tr = Translator.from_pretrained(None if MODELS is None else f"{MODELS}/nllb")
tr.transliterate(blocks)
for b in blocks:
    print(f"\nblock {b.block} transliteration: {b.transliteration}")

In [ ]:
# correct the transliteration here if needed, e.g. blocks[0].transliteration = "..."
tr.translate(blocks, "en")
for b in blocks:
    print(f"block {b.block}: {b.translation['en']}")
save_blocks(blocks, "blocks.json")

## 4. Text without an image

Type Gardiner codes (TLA spelling, space-separated) and translate directly.

In [ ]:
from akhet import TextBlock
typed = [TextBlock(hieroglyphs="V30 N16 N16")]
tr.translate(tr.transliterate(typed), "en")
print(typed[0].transliteration, "|", typed[0].translation["en"])